Now that you've deployed your endpoint - it's time to slam it!

In [1]:
import os
import getpass
from dotenv import load_dotenv

load_dotenv()

if not os.environ.get("FIREWORKS_API_KEY"):
    os.environ["FIREWORKS_API_KEY"] = getpass.getpass("Enter your Fireworks API key: ")

Let's try with 1 request, just to verify our endpoint is alive.

Make sure you provide your own endpoint identifier! It will look something like this:

- `accounts/fireworks/models/gpt-oss-20b` (serverless)
- Or your custom on-demand deployment identifier

In [2]:
from langchain_fireworks import ChatFireworks

# REPLACE WITH YOUR OWN ENDPOINT IDENTIFIER
model_endpoint = "accounts/fireworks/models/gpt-oss-20b"

llm = ChatFireworks(model=model_endpoint)

response = llm.invoke("How much wood could a wood chuck chuck if a wood chuck could chuck wood?")
print(response.content)

Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x111d0a0d0>
Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x111d08b90>


It’s a tongue‑twisting legend, but over the years people have tried to make it a serious question.  In one of the most famous modern “research” accounts, Randy McKenzie (the University of Florida’s chief squirrel‑specialist, which is a legit animal‑behavioural unit) estimated that a burrowing woodchuck could chuck roughly **700 pounds of wood** in a day—roughly the weight of a small car.  That’s about **3 cubic yards of spruce, fir or pine** if you work it out in volume.

Some other studies have taken a more skeptical stance.  In 2017 the University of Arkansas hunters, in a tongue‑in‑cheek “official” paper, calculated that a woodchuck’s digging “effort” could move about **125–140 stones of earth** per day, but that’s not wood at all – so technically, if a woodchuck could literally move wood rather than soil, the figure would be in the same ballpark.

In lay terms, the short answer is: **a woodchuck could probably chuck anywhere from 500 to 1,000 pounds of wood each day**—if it were ac

Now, let's SLAM IT.

In [4]:
import asyncio

async def send_request(llm, idx):
    try:
        response = await llm.ainvoke(
            f"How much wood could a wood chuck chuck if a wood chuck could chuck wood? (Request {idx})"
        )
        print(f"Response {idx}: {response.content[:60]}...")
    except Exception as e:
        print(f"Request {idx} failed: {e}")

async def main():
    tasks = [send_request(llm, i) for i in range(1, 25)]
    await asyncio.gather(*tasks)

await main()

Request 2 failed: {"error":{"message":"You have exceeded your rate limit for this API. Please try again later. For more information, see https://docs.fireworks.ai/guides/quotas_usage/rate-limits.","param":null,"code":"RATE_LIMIT_EXCEEDED","type":"error"},"request_id":"chatcmpl-bb659c84e8ca4822a1ecdbf4a2ce033b"}
Request 14 failed: {"error":{"message":"You have exceeded your rate limit for this API. Please try again later. For more information, see https://docs.fireworks.ai/guides/quotas_usage/rate-limits.","param":null,"code":"RATE_LIMIT_EXCEEDED","type":"error"},"request_id":"chatcmpl-c58fb0e410024a228adcf752ba5f62e1"}
Request 23 failed: {"error":{"message":"You have exceeded your rate limit for this API. Please try again later. For more information, see https://docs.fireworks.ai/guides/quotas_usage/rate-limits.","param":null,"code":"RATE_LIMIT_EXCEEDED","type":"error"},"request_id":"chatcmpl-d31a9f604b1840b2a07f7c179e1df178"}
Request 18 failed: {"error":{"message":"You have exceeded y

# Activity 1

Use RAGAS to evaluate your open-source Fireworks AI powered RAG app against an OpenAI gpt-4.1-mini powered equivalent. Compare retrieval quality, answer faithfulness, and end-to-end accuracy across both providers.

In [41]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:

import nest_asyncio
from dotenv import load_dotenv
from langchain_classic.retrievers.ensemble import EnsembleRetriever
from langchain_community.retrievers import BM25Retriever
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
    
#Semantic search scoped to a single case
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.llms import LangchainLLMWrapper
from ragas.testset import TestsetGenerator
from langchain_community.document_loaders import DirectoryLoader, PyMuPDFLoader
import tiktoken

nest_asyncio.apply()  


In [5]:

data_dir = './data'
directory_loader = DirectoryLoader(
    data_dir, glob="**/*.pdf", loader_cls=PyMuPDFLoader
)
documents = directory_loader.load()
def _tiktoken_len(text: str) -> int:
    """Return token length using tiktoken; used for chunk length measurement."""
    tokens = tiktoken.encoding_for_model("gpt-4o").encode(text)
    return len(tokens)\
    

# Split documents
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=750, chunk_overlap=0, length_function=_tiktoken_len)

chunks = text_splitter.split_documents(documents) if documents else []

In [6]:
generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-mini"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())

/var/folders/bz/qcwzc_9j6zgggtsc_fbjryy40000gp/T/ipykernel_4319/22872021.py:1: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-mini"))
/var/folders/bz/qcwzc_9j6zgggtsc_fbjryy40000gp/T/ipykernel_4319/22872021.py:2: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())


In [ ]:
from langchain_core.messages import AIMessage, HumanMessage, ToolMessage
from app.graphs.simple_agent import graph, build_graph

from ragas.metrics import LLMContextRecall, Faithfulness, FactualCorrectness, ResponseRelevancy, ContextEntityRecall, NoiseSensitivity
from ragas.testset.graph import KnowledgeGraph, Node, NodeType
from ragas.testset.transforms import apply_transforms, Parallel
from ragas.testset.transforms.extractors import HeadlinesExtractor, SummaryExtractor, EmbeddingExtractor
from ragas.testset.transforms.extractors.llm_based import NERExtractor, ThemesExtractor
from ragas.testset.transforms.splitters import HeadlineSplitter
from ragas.testset.transforms.relationship_builders import CosineSimilarityBuilder, OverlapScoreBuilder
from ragas.testset.transforms.filters import CustomNodeFilter
from ragas.testset import TestsetGenerator
from ragas.testset.synthesizers import SingleHopSpecificQuerySynthesizer
from ragas.utils import num_tokens_from_string
from ragas import EvaluationDataset, evaluate, RunConfig
from ragas import 
from ragas.llms import LangchainLLMWrapper
import copy
from ragas.metrics import (
  ContextEntityRecall,
  ContextPrecision,
  FactualCorrectness,
  Faithfulness,
  LLMContextRecall

# Filter that matches HeadlinesExtractor's filter
def long_doc_filter(node):
    return node.type == NodeType.DOCUMENT and num_tokens_from_string(node.properties.get("page_content", "")) > 500

def chunk_filter(node):
    return node.type == NodeType.CHUNK

# Build KG
kg = KnowledgeGraph()
for doc in documents:
    kg.nodes.append(Node(
        type=NodeType.DOCUMENT,
        properties={"page_content": doc.page_content, "document_metadata": doc.metadata}
    ))

# Build transforms with matching filter on HeadlineSplitter
transforms = [
    HeadlinesExtractor(llm=generator_llm, filter_nodes=long_doc_filter),
    HeadlineSplitter(min_tokens=500, filter_nodes=long_doc_filter),  # <-- fix: same filter
    SummaryExtractor(llm=generator_llm, filter_nodes=long_doc_filter),
    CustomNodeFilter(llm=generator_llm, filter_nodes=chunk_filter),
    Parallel(
        EmbeddingExtractor(embedding_model=generator_embeddings, property_name="summary_embedding", embed_property_name="summary", filter_nodes=long_doc_filter),
        ThemesExtractor(llm=generator_llm, filter_nodes=chunk_filter),
        NERExtractor(llm=generator_llm, filter_nodes=chunk_filter),
    ),
    Parallel(
        CosineSimilarityBuilder(property_name="summary_embedding", new_property_name="summary_similarity", threshold=0.7, filter_nodes=long_doc_filter),
        OverlapScoreBuilder(threshold=0.01, filter_nodes=chunk_filter),
    ),
]

apply_transforms(kg, transforms)

generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings, knowledge_graph=kg)

query_distribution = [(SingleHopSpecificQuerySynthesizer(llm=generator_llm), 1.0)]
dataset = generator.generate(testset_size=10, query_distribution=query_distribution)


Applying HeadlinesExtractor:   0%|          | 0/17 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/17 [00:00<?, ?it/s]

Applying SummaryExtractor:   0%|          | 0/18 [00:00<?, ?it/s]

Property 'summary' already exists in node 'bc5c32'. Skipping!


Applying CustomNodeFilter:   0%|          | 0/42 [00:00<?, ?it/s]

...

Applying CosineSimilarityBuilder:   0%|          | 0/1 [00:00<?, ?it/s]

Applying OverlapScoreBuilder:   0%|          | 0/1 [00:00<?, ?it/s]

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/1 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/10 [00:00<?, ?it/s]

In [10]:
dataset.to_pandas()

,user_input,reference_contexts,reference,persona_name,query_style,query_length,synthesizer_name
0,where The Ohio State University in the guideli...,[VETERINARY PRACTICE GUIDELINES 2021 AAHA/AAFP...,The Ohio State University is listed as one of ...,New Kitten Owner,POOR_GRAMMAR,LONG,single_hop_specific_query_synthesizer
1,Can you explane what the 2021 AAFP End of Life...,"[their cat’s maturation and aging process, and...",The 2021 AAFP End of Life Online Educational T...,Kitten Care Educator,MISSPELLED,LONG,single_hop_specific_query_synthesizer
2,What age range defines a mature adult cat?,[Importance of Feline-Friendly Handling\nBoth ...,A mature adult cat is defined as being from 7 ...,New Kitten Owner,WEB_SEARCH_LIKE,SHORT,single_hop_specific_query_synthesizer
3,Could you explain what the AAFP provides in te...,[Feline-Friendly Strategies\nFeline-friendly h...,The AAFP offers detailed feline-friendly handl...,New Kitten Owner,PERFECT_GRAMMAR,LONG,single_hop_specific_query_synthesizer
4,How do veterinarian decide if old cats need sp...,"[For example, some senior cats aged 10 years a...",Some senior cats aged 10 years and older may r...,New Kitten Owner,POOR_GRAMMAR,MEDIUM,single_hop_specific_query_synthesizer
5,Why is understanding the human–cat bond import...,[Lifestyle Risk Assessment\nUnderstanding the ...,Understanding the human–cat bond is essential ...,New Kitten Owner,PERFECT_GRAMMAR,MEDIUM,single_hop_specific_query_synthesizer
6,What are the important health considerations a...,[detection of changes and identiﬁcation of tre...,"During veterinary examinations of kittens, imp...",New Kitten Owner,WEB_SEARCH_LIKE,LONG,single_hop_specific_query_synthesizer
7,What are importent signs to watch for in senoi...,[Mature Adult and Senior Cats\nThe medical his...,"In senior cats, the medical history and examin...",Kitten Care Educator,MISSPELLED,MEDIUM,single_hop_specific_query_synthesizer
8,What is a musculoskeltal examanation and why i...,[Detecting signs of pain or anxiety and evalua...,A detailed musculoskeletal examination is crit...,New Kitten Owner,MISSPELLED,MEDIUM,single_hop_specific_query_synthesizer
9,How can understanding the natural behaviors an...,[Understanding and Enhancing Behavior by Life ...,Understanding the natural behaviors and social...,Kitten Care Educator,PERFECT_GRAMMAR,LONG,single_hop_specific_query_synthesizer


In [11]:
dataset.to_pandas().to_csv('cat_care_sdg.csv')

In [ ]:
# first test serverless GPT OSS 20b on Fireworks.ai. (this is the default model in the app import)

custom_run_config = RunConfig(timeout=360)
evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1"))

oss_rag_dataset = copy.deepcopy(dataset)


/var/folders/bz/qcwzc_9j6zgggtsc_fbjryy40000gp/T/ipykernel_4319/318958530.py:4: DeprecationWarning: Importing ContextEntityRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ContextEntityRecall
  from ragas.metrics import (
/var/folders/bz/qcwzc_9j6zgggtsc_fbjryy40000gp/T/ipykernel_4319/318958530.py:4: DeprecationWarning: Importing ContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ContextPrecision
  from ragas.metrics import (
/var/folders/bz/qcwzc_9j6zgggtsc_fbjryy40000gp/T/ipykernel_4319/318958530.py:4: DeprecationWarning: Importing FactualCorrectness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import FactualCorrectness
  from ragas.metrics 

In [14]:
graph = build_graph().compile()

In [23]:
import time
from app.rag import build_graph

rag_graph = build_graph()

for test_row in oss_rag_dataset:
    response = rag_graph.invoke({"question": test_row.eval_sample.user_input})
    test_row.eval_sample.response = response["response"]
    test_row.eval_sample.retrieved_contexts = [context.page_content for context in response["context"]]
    time.sleep(2)  # To try to avoid rate limiting.


In [ ]:
oss_evaluation_dataset = EvaluationDataset.from_pandas(oss_rag_dataset.to_pandas())
evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1"))

/var/folders/bz/qcwzc_9j6zgggtsc_fbjryy40000gp/T/ipykernel_4319/1484578614.py:6: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1"))


In [ ]:

custom_run_config = RunConfig(timeout=360)

oss_result = evaluate(
    dataset=oss_evaluation_dataset,
    metrics=[LLMContextRecall(), Faithfulness(), FactualCorrectness(), ResponseRelevancy(), ContextEntityRecall(), NoiseSensitivity()],
    llm=evaluator_llm,
    run_config=custom_run_config
)
oss_result

/var/folders/bz/qcwzc_9j6zgggtsc_fbjryy40000gp/T/ipykernel_4319/4084518634.py:1: DeprecationWarning: Importing LLMContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import LLMContextRecall
  from ragas.metrics import LLMContextRecall, Faithfulness, FactualCorrectness, ResponseRelevancy, ContextEntityRecall, NoiseSensitivity
/var/folders/bz/qcwzc_9j6zgggtsc_fbjryy40000gp/T/ipykernel_4319/4084518634.py:1: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import LLMContextRecall, Faithfulness, FactualCorrectness, ResponseRelevancy, ContextEntityRecall, NoiseSensitivity
/var/folders/bz/qcwzc_9j6zgggtsc_fbjryy40000gp/T/ipykernel_4319/4084518634.py:1: DeprecationWarning: Importing FactualCorrectn

Evaluating:   0%|          | 0/60 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


{'context_recall': 1.0000, 'faithfulness': 0.5843, 'factual_correctness(mode=f1)': 0.3010, 'answer_relevancy': 0.8600, 'context_entity_recall': 0.4575, 'noise_sensitivity(mode=relevant)': 0.1684}

In [26]:
oss= {'context_recall': 1.0000, 'faithfulness': 0.5843, 'factual_correctness(mode=f1)': 0.3010, 'answer_relevancy': 0.8600, 'context_entity_recall': 0.4575, 'noise_sensitivity(mode=relevant)': 0.1684}

In [ ]:
# OpenAI GPT closed source 
rag_graph2 = build_graph(model_name='gpt-4.1-mini')


In [20]:
openai_rag_dataset = copy.deepcopy(dataset)

for test_row in openai_rag_dataset:
    response = rag_graph2.invoke({"question": test_row.eval_sample.user_input})
    test_row.eval_sample.response = response["response"]
    test_row.eval_sample.retrieved_contexts = [context.page_content for context in response["context"]]
    time.sleep(2)  # To try to avoid rate limiting.

In [ ]:

evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1"))
evaluation_dataset = EvaluationDataset.from_pandas(openai_rag_dataset.to_pandas())


/var/folders/bz/qcwzc_9j6zgggtsc_fbjryy40000gp/T/ipykernel_4319/2173580469.py:4: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1"))


In [ ]:

openai_result = evaluate(
    dataset=evaluation_dataset,
    metrics=[LLMContextRecall(), Faithfulness(), FactualCorrectness(), ResponseRelevancy(), ContextEntityRecall(), NoiseSensitivity()],
    llm=evaluator_llm,
    run_config=custom_run_config
)
openai_result

/var/folders/bz/qcwzc_9j6zgggtsc_fbjryy40000gp/T/ipykernel_4319/2632495459.py:1: DeprecationWarning: Importing LLMContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import LLMContextRecall
  from ragas.metrics import LLMContextRecall, Faithfulness, FactualCorrectness, ResponseRelevancy, ContextEntityRecall, NoiseSensitivity
/var/folders/bz/qcwzc_9j6zgggtsc_fbjryy40000gp/T/ipykernel_4319/2632495459.py:1: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import LLMContextRecall, Faithfulness, FactualCorrectness, ResponseRelevancy, ContextEntityRecall, NoiseSensitivity
/var/folders/bz/qcwzc_9j6zgggtsc_fbjryy40000gp/T/ipykernel_4319/2632495459.py:1: DeprecationWarning: Importing FactualCorrectn

Evaluating:   0%|          | 0/60 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Task exception was never retrieved
future: <Task finished name='Task-117' coro=<as_completed.<locals>.sema_coro() done, defined at /Users/stevegoodman/dev/AIE9/16_LLM_Servers/.venv/lib/python3.13/site-packages/ragas/async_utils.py:75> exception=KeyboardInterrupt()>
Traceback (most recent call last):
  File "/Users/stevegoodman/dev/AIE9/16_LLM_Servers/.venv/lib/python3.13/site-packages/IPython/core/interactiveshell.py", line 3747, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
    ~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/var/folders/bz/qcwzc_9j6zgggtsc_fbjryy40000gp/T/ipykernel_4319/313868771.py", line 44, in <module>
    apply_transforms(kg, transforms)
    ~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^
  File "/Users/ste

{'context_recall': 0.8667, 'faithfulness': 0.6875, 'factual_correctness(mode=f1)': 0.6360, 'answer_relevancy': 0.5959, 'context_entity_recall': 0.4927, 'noise_sensitivity(mode=relevant)': 0.1956}

In [27]:
gpt4_1_mini={'context_recall': 0.8667, 'faithfulness': 0.6875, 'factual_correctness(mode=f1)': 0.6360, 'answer_relevancy': 0.5959, 'context_entity_recall': 0.4927, 'noise_sensitivity(mode=relevant)': 0.1956}

In [ ]:
# finally the dedicated endpoint on fireworks.ai - still GPT oss 20B running at $4 hr
dedictated_endpoint = 'accounts/stevejgoodman-nnh4io/deployments/pbctr4p4'
rag_graph3 = build_graph(model_name=dedictated_endpoint)


In [35]:
oss_dedicated_rag_dataset = copy.deepcopy(dataset)

for test_row in oss_dedicated_rag_dataset:
    response = rag_graph3.invoke({"question": test_row.eval_sample.user_input})
    test_row.eval_sample.response = response["response"]
    test_row.eval_sample.retrieved_contexts = [context.page_content for context in response["context"]]
    time.sleep(2)  # To try to avoid rate limiting.

In [36]:
evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1"))
evaluation_dataset = EvaluationDataset.from_pandas(oss_dedicated_rag_dataset.to_pandas())

/var/folders/bz/qcwzc_9j6zgggtsc_fbjryy40000gp/T/ipykernel_4319/1638270924.py:1: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1"))


In [37]:
oss_dedicated_result = evaluate(
    dataset=evaluation_dataset,
    metrics=[LLMContextRecall(), Faithfulness(), FactualCorrectness(), ResponseRelevancy(), ContextEntityRecall(), NoiseSensitivity()],
    llm=evaluator_llm,
    run_config=custom_run_config
)
oss_dedicated_result

Evaluating:   0%|          | 0/60 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


{'context_recall': 0.8667, 'faithfulness': 0.5264, 'factual_correctness(mode=f1)': 0.3970, 'answer_relevancy': 0.7544, 'context_entity_recall': 0.4604, 'noise_sensitivity(mode=relevant)': 0.2090}

In [38]:
oss_dedicated = {'context_recall': 0.8667, 'faithfulness': 0.5264, 'factual_correctness(mode=f1)': 0.3970, 'answer_relevancy': 0.7544, 'context_entity_recall': 0.4604, 'noise_sensitivity(mode=relevant)': 0.2090}

In [39]:
metrics = list(oss.keys())
import pandas as pd
pd.DataFrame([oss, oss_dedicated, gpt4_1_mini], index = ['gpt_oss_20_serverless', 'gpt_oss_20_dedicated' ,'gpt4.1-mini'])

,context_recall,faithfulness,factual_correctness(mode=f1),answer_relevancy,context_entity_recall,noise_sensitivity(mode=relevant)
gpt_oss_20_serverless,1.0000,0.5843,0.301,0.8600,0.4575,0.1684
gpt_oss_20_dedicated,0.8667,0.5264,0.397,0.7544,0.4604,0.2090
gpt4.1-mini,0.8667,0.6875,0.636,0.5959,0.4927,0.1956


# Conclusion

## Rag Metrics
GPT 4.1 Mini wins on answer quality (factual correctness, faithfulness). 

2 versions of OSS (serverless and dedicated endpoint)
OSS20b  wins on relevancy, it's more concise and stays on topic but  gets  facts wrong. 

Caveat is my  test dataset size is small. 



## Latency & Cost



Serverless is about 1 5x less expensive than gpt4.1-mini, even thouh more tokens were processed.
However, dedicated endpoint that runs for about 1m 20sec, ($4 an hr) is more expensive than gpt4.1-mini by  roughly 6x. 

For the short period we're running it, and 1 example running at a time, we're not really maximising use of the compute resources, (rag so we're spending some of that time goint to retreive chunks etc).

| MODEL | Latency_total(sec) | latency_per_completion | Total Cost $ |
| ---- | ----- | ------ | ---- |
| gpt_oss_20_serverless | 92 | 9 | 0.003 |
| gpt_oss_20_dedicated | 79 | 8 | 0.014 |
| gpt4.1-mini |	27 | 3 | 0.090 |